[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S23_marketing_analytics.ipynb)

# Sesión 23 · Marketing analytics

**Módulo 6: Negocio y extras** · ⏱️ Duración estimada: 60 a 75 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Construir un embudo de conversión y ubicar dónde se pierden más clientes.
2. Medir el efecto real de una campaña con un grupo de control: incremento y lift.
3. Segmentar clientes con RFM (recencia, frecuencia y monto).
4. Calcular el ROI de una campaña con supuestos explícitos y ver cuánto dependen de ellos.

## 📋 Qué debes saber antes
Módulo 3 (pandas: `groupby`, fechas, `pd.qcut`), funciones (sesión 3) y gráficos de barras (sesiones 14 y 15).

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})


# ---------- Datos 1: campaña de correo de una cadena de tiendas, con grupo de control ----------
_n = 20000
_grupo = np.where(rng.random(_n) < 0.2, "control", "campaña")
_seg = rng.choice(["nuevo", "regular", "frecuente"], _n, p=[0.3, 0.45, 0.25])
_canal = rng.choice(["app", "web", "tienda"], _n, p=[0.45, 0.35, 0.2])
_base = pd.Series(_seg).map({"nuevo": 0.02, "regular": 0.045, "frecuente": 0.09}).to_numpy()
_efecto = pd.Series(_seg).map({"nuevo": 0.03, "regular": 0.01, "frecuente": 0.002}).to_numpy()
_ticket = pd.Series(_seg).map({"nuevo": 110, "regular": 130, "frecuente": 160}).to_numpy()
_organica = rng.random(_n) < _base
_en_campana = _grupo == "campaña"
_abrio = _en_campana & (rng.random(_n) < np.where(_organica, 0.6, 0.3) + 0.1 * (_canal == "app"))
_clic = _abrio & (rng.random(_n) < 0.25)
_compro = _organica | (_clic & (rng.random(_n) < _efecto / 0.08))
campana = pd.DataFrame({
    "id_cliente": np.arange(1, _n + 1), "grupo": _grupo, "segmento": _seg, "canal": _canal,
    "abrio": _abrio.astype(int), "clic": _clic.astype(int), "compro": _compro.astype(int),
    "monto": np.where(_compro, np.round(rng.lognormal(np.log(_ticket), 0.5), 2), 0.0),
})
ETAPAS = ["recibió", "abrió", "clic", "compró"]
SEGMENTOS = ["nuevo", "regular", "frecuente"]
SUPUESTOS = {"margen": 0.30, "costo_contacto": 0.15, "descuento": 0.10}

# ---------- Datos 2: compras de 1500 clientes en el último año ----------
FECHA_CORTE = pd.Timestamp("2025-06-30")
_filas = []
for _c in range(1, 1501):
    _fin = int(rng.integers(90, 330)) if rng.random() < 0.3 else int(rng.integers(0, 60))
    _veces = 1 + rng.poisson(rng.gamma(1.5, 0.8) * (364 - _fin) / 30)
    for _d in rng.integers(_fin, 365, _veces):
        _filas.append((_c, FECHA_CORTE - pd.Timedelta(days=int(_d)), round(float(rng.lognormal(np.log(80), 0.6)), 2)))
transacciones = pd.DataFrame(_filas, columns=["id_cliente", "fecha", "monto"]).sort_values(["fecha", "id_cliente"], kind="stable").reset_index(drop=True)

_D = copy.deepcopy({"campana": campana, "transacciones": transacciones})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _camp():
    return [f for f in campana.itertuples(index=False) if f.grupo == "campaña"]


def _tasa(filas):
    return math.fsum(f.compro for f in filas) / len(filas)


def _embudo_ref():
    c = _camp()
    cuentas = [len(c), sum(f.abrio for f in c), sum(f.clic for f in c), sum(f.clic and f.compro for f in c)]
    return [[cuentas[i], 1.0 if i == 0 else cuentas[i] / cuentas[i - 1], cuentas[i] / cuentas[0]] for i in range(4)]


def _por_grupo(seg=None):
    filas = [f for f in campana.itertuples(index=False) if seg is None or f.segmento == seg]
    c = [f for f in filas if f.grupo == "campaña"]
    k = [f for f in filas if f.grupo == "control"]
    return c, k


def _roi_ref(n, tc, tk, ticket, margen, costo_contacto, descuento):
    """ROI recalculado por cliente contactado."""
    if n == 0:
        return None
    ganancia_por_cliente = (tc - tk) * ticket * margen
    costo_por_cliente = costo_contacto + tc * ticket * descuento
    if costo_por_cliente == 0:
        return None
    return ganancia_por_cliente / costo_por_cliente - 1


def _rfm_ref(trans, corte):
    datos = {}
    for f in trans.itertuples(index=False):
        d = datos.setdefault(f.id_cliente, [None, 0, 0.0])
        d[0] = f.fecha if d[0] is None or f.fecha > d[0] else d[0]
        d[1] += 1
        d[2] += f.monto
    return {c: ((corte - d[0]).days, d[1], d[2]) for c, d in sorted(datos.items())}


def _quintil(valores, mayor_es_mejor):
    """Puntaje 1 a 5 por rango (empates por orden de aparición), recalculado sin pandas."""
    n = len(valores)
    orden = sorted(range(n), key=lambda i: (valores[i] if mayor_es_mejor else -valores[i], i))
    bordes = [1 + (n - 1) * q / 5 for q in range(6)]
    puntaje = [0] * n
    for rango, i in enumerate(orden, start=1):
        puntaje[i] = next(k for k in range(1, 6) if rango <= bordes[k] + 1e-9)
    return puntaje


def _segmento(r, f):
    if r >= 4 and f >= 4:
        return "campeones"
    if r <= 2 and f >= 3:
        return "en riesgo"
    if r >= 4 and f <= 2:
        return "nuevos"
    if r <= 2 and f <= 2:
        return "dormidos"
    return "regulares"


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    _sin_cambios_df(r, "campana")
    _df(r, "embudo", ["clientes", "conv_paso", "conv_total"], _embudo_ref(),
        "cuenta los clientes del grupo campaña en cada etapa (\"compró\" = compró **y** hizo clic) y divide por la etapa anterior y por la primera", indice=ETAPAS, tol=1e-9)
    ax = _grafico(r, "ax_embudo")
    if ax is not None:
        cuentas = [f[0] for f in _embudo_ref()]
        barras = _barras(ax)
        anchos = [b.get_width() for b in barras]
        if len(barras) != 4:
            r.mal(f"`ax_embudo` tiene {len(barras)} barras y se esperaban 4, una por etapa.")
        elif _cerca_lista(anchos, cuentas):
            r.mal("En `ax_embudo`, la primera etapa quedó abajo: invierte el orden para que \"recibió\" quede arriba.")
        elif not _cerca_lista(anchos, cuentas[::-1]):
            r.mal("Las barras de `ax_embudo` deberían ser horizontales y medir la columna `clientes` de `embudo`.")
        else:
            r.ok("`ax_embudo` muestra el embudo de arriba abajo.")
        _rotulos(r, "ax_embudo", ax, None, "Clientes", None)
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_pct_clic": "b6ed29611b1a400e7cffd9f025ef3e89be3345dce273e7c27e6157137bd1a608",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    c, k = _por_grupo()
    tc, tk = _tasa(c), _tasa(k)
    _ser(r, "tasas", [tc, tk], "la tasa de compra (`compro`) de cada grupo, con índice campaña y control", indice=["campaña", "control"], tol=1e-9)
    _esc(r, "lift", tc / tk, "la tasa de la campaña dividida por la del control", tol=1e-9)
    _esc(r, "incremento_pp", (tc - tk) * 100, "la diferencia de tasas, en puntos porcentuales", tol=1e-9)
    _esc(r, "compras_incrementales", (tc - tk) * len(c), "el incremento de tasa por la cantidad de clientes del grupo campaña", tol=1e-6)
    _esc(r, "compras_atribuidas", sum(f.clic and f.compro for f in c), "los clientes del grupo campaña que hicieron clic y compraron", tol=0)
    filas = []
    for s in SEGMENTOS:
        cs, ks = _por_grupo(s)
        a, b = _tasa(ks), _tasa(cs)
        filas.append([a, b, (b - a) * 100, b / a])
    _df(r, "por_segmento", ["tasa_control", "tasa_campana", "incremento_pp", "lift"], filas,
        "una fila por segmento en el orden de `SEGMENTOS`", indice=SEGMENTOS, tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_atribuidas_mayor": "6c5fb3b25e6ba7dcf12155440e0c51b36a0492e24123968a10f8c31c304ebb6e",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    _sin_cambios_df(r, "transacciones")
    f = r.funcion("calcular_rfm")
    if f is not _FALTA:
        corte = pd.Timestamp("2025-01-31")
        prueba = pd.DataFrame({"id_cliente": [7, 3, 7, 3, 9, 7],
                               "fecha": pd.to_datetime(["2025-01-02", "2024-12-01", "2025-01-31", "2024-12-01", "2024-02-01", "2025-01-10"]),
                               "monto": [10.0, 20.5, 5.0, 4.5, 100.0, 0.0]})
        casos = [("con clientes de varias compras, dos compras el mismo día y una compra en la fecha de corte", prueba),
                 ("con un solo cliente y una sola compra", prueba.iloc[[4]]),
                 ("sin transacciones", prueba.iloc[0:0])]
        for texto, df in casos:
            original = df.copy()
            try:
                res = f(df, corte)
            except Exception as ex:
                r.mal(f"`calcular_rfm` {texto} lanzó {type(ex).__name__}: {ex}")
                continue
            if not df.equals(original):
                r.mal("`calcular_rfm` modificó el DataFrame que recibió.")
                continue
            ref = _rfm_ref(df, corte)
            if not isinstance(res, pd.DataFrame) or [str(c) for c in res.columns] != ["recencia", "frecuencia", "monetario"]:
                r.mal("`calcular_rfm` debería devolver un DataFrame con las columnas recencia, frecuencia y monetario, en ese orden.")
                break
            if [_norm(i) for i in res.index] != list(ref):
                r.mal(f"`calcular_rfm` {texto} debería tener una fila por cliente, con `id_cliente` como índice ordenado.")
                continue
            if not all(_mismo(a, b[0]) and _mismo(c, b[1]) and _mismo(d, b[2], 1e-6)
                       for (a, c, d), b in zip(res.itertuples(index=False), ref.values())):
                r.mal(f"`calcular_rfm` {texto} da valores incorrectos; la recencia son los días entre la última compra y la fecha de corte (`.dt.days`).")
            else:
                r.ok(f"`calcular_rfm` funciona {texto}.")
    ref = _rfm_ref(transacciones, FECHA_CORTE)
    rec = [v[0] for v in ref.values()]
    fre = [v[1] for v in ref.values()]
    mon = [v[2] for v in ref.values()]
    R, F, M = _quintil(rec, False), _quintil(fre, True), _quintil(mon, True)
    rfm_v = r.var("rfm")
    if rfm_v is not _FALTA:
        cols = ["recencia", "frecuencia", "monetario", "R", "F", "M", "segmento"]
        if not isinstance(rfm_v, pd.DataFrame) or [str(c) for c in rfm_v.columns] != cols:
            r.mal(f"`rfm` debería ser un DataFrame con las columnas {cols}, en ese orden.")
        elif [_norm(i) for i in rfm_v.index] != list(ref):
            r.mal("`rfm` debería tener una fila por cliente, con `id_cliente` como índice ordenado.")
        elif not (_cerca_lista(rfm_v["recencia"], rec) and _cerca_lista(rfm_v["frecuencia"], fre) and _cerca_lista(rfm_v["monetario"], mon)):
            r.mal("Las columnas recencia, frecuencia y monetario de `rfm` no coinciden: usa `calcular_rfm(transacciones, FECHA_CORTE)`.")
        else:
            bien = True
            for nombre, ref_p in (("R", R), ("F", F), ("M", M)):
                if [int(x) for x in rfm_v[nombre]] != ref_p:
                    r.mal(f"La columna `{nombre}` de `rfm` no coincide: usa `pd.qcut` sobre el `rank(method=\"first\")`, con 5 = mejor.")
                    bien = False
            if bien:
                r.ok("Los puntajes R, F y M de `rfm` son correctos.")
                segs = [_segmento(a, b) for a, b in zip(R, F)]
                if [str(s) for s in rfm_v["segmento"]] != segs:
                    r.mal("La columna `segmento` de `rfm` no coincide: aplica las reglas en el orden indicado.")
                else:
                    r.ok("La columna `segmento` de `rfm` es correcta.")
                    cuenta = {s: segs.count(s) for s in set(segs)}
                    orden = sorted(cuenta, key=lambda s: (-cuenta[s], s))
                    _df(r, "resumen_rfm", ["clientes", "monetario_medio"],
                        [[cuenta[s], statistics.fmean([m for m, g in zip(mon, segs) if g == s])] for s in orden],
                        "clientes y monetario medio por segmento, de más a menos clientes", indice=orden, tol=1e-6)
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_recencia_mejor": "cfd6d8112dc41f01f4e37fc8784e02e18a4dbd6596684d84860bdbcd7bd7210f",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    f = r.funcion("roi_campana")
    if f is not _FALTA:
        casos = [((1000, 0.05, 0.03, 100.0, 0.3, 0.2, 0.1), "una campaña con efecto"),
                 ((1000, 0.04, 0.04, 100.0, 0.3, 0.2, 0.1), "una campaña sin efecto"),
                 ((1000, 0.03, 0.04, 100.0, 0.3, 0.2, 0.1), "una campaña que empeora las compras"),
                 ((500, 0.06, 0.02, 80.0, 0.4, 0.0, 0.0), "una campaña sin costo de contacto ni descuento"),
                 ((0, 0.05, 0.03, 100.0, 0.3, 0.2, 0.1), "cero clientes contactados")]
        for args, texto in casos:
            esperado = _roi_ref(*args)
            r.caso(f"roi_campana{args}", f, args, esperado=esperado, tol=1e-9,
                   motivo=f"revisa la fórmula con {texto}" + (" (sin costo, el ROI no se puede calcular: devuelve None)" if esperado is None else ""))
    c, k = _por_grupo()
    tc, tk = _tasa(c), _tasa(k)
    ticket = statistics.fmean([x.monto for x in c if x.compro])
    _esc(r, "ticket", ticket, "el monto promedio de las compras del grupo campaña", tol=1e-6)
    _esc(r, "roi_total", _roi_ref(len(c), tc, tk, ticket, **SUPUESTOS), "`roi_campana` con los datos de la campaña y `SUPUESTOS`", tol=1e-9)
    margenes, costos = [0.2, 0.3, 0.4, 0.5], [0.05, 0.15, 0.3]
    filas = [[_roi_ref(len(c), tc, tk, ticket, m, cc, SUPUESTOS["descuento"]) for cc in costos] for m in margenes]
    v = r.var("sensibilidad")
    if v is not _FALTA:
        if not isinstance(v, pd.DataFrame) or v.shape != (4, 3):
            r.mal("`sensibilidad` debería ser un DataFrame de 4 filas (márgenes) y 3 columnas (costos de contacto).")
        elif not _cerca_lista([float(i) for i in v.index], margenes) or not _cerca_lista([float(x) for x in v.columns], costos):
            r.mal("`sensibilidad` debería tener los márgenes como índice y los costos de contacto como columnas.")
        elif not all(_cerca_lista(a, b, 1e-9) for a, b in zip(v.to_numpy(float).tolist(), filas)):
            r.mal("Los valores de `sensibilidad` no coinciden: cada celda es el ROI con ese margen y ese costo de contacto (el descuento queda fijo).")
        else:
            r.ok("`sensibilidad` es correcta.")
    s = r.var("supuestos_campana", list)
    if s is not _FALTA:
        if len(s) < 3 or not all(isinstance(x, str) and len(x.strip()) >= 15 for x in s):
            r.mal("Escribe en `supuestos_campana` al menos 3 supuestos, cada uno como una frase completa.")
        else:
            r.ok("Dejaste tus supuestos por escrito.")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_roi_sin_efecto": "b6dce671d8b263f7d6f809dff836fbeeff5cc13c54454cc5700b741c25faddfc",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    ganancias, costos, rois = {}, {}, {}
    for s in SEGMENTOS:
        cs, ks = _por_grupo(s)
        tc, tk = _tasa(cs), _tasa(ks)
        t = statistics.fmean([x.monto for x in cs if x.compro])
        ganancias[s] = len(cs) * (tc - tk) * t * SUPUESTOS["margen"]
        costos[s] = len(cs) * (SUPUESTOS["costo_contacto"] + tc * t * SUPUESTOS["descuento"])
        rois[s] = ganancias[s] / costos[s] - 1
    _ser(r, "roi_por_segmento", [rois[s] for s in SEGMENTOS], "el ROI de cada segmento con su propio ticket y sus propias tasas, en el orden de `SEGMENTOS`", indice=SEGMENTOS, tol=1e-9)
    buenos = [s for s in SEGMENTOS if rois[s] > 0]
    v = r.var("segmentos_rentables", list)
    if v is not _FALTA:
        if v != buenos:
            r.mal("`segmentos_rentables` debería listar, en el orden de `SEGMENTOS`, los segmentos con ROI positivo.")
        else:
            r.ok("`segmentos_rentables` es correcto.")
    if buenos:
        _esc(r, "roi_focalizado", sum(ganancias[s] for s in buenos) / sum(costos[s] for s in buenos) - 1,
             "la suma de ganancias incrementales de los segmentos rentables dividida por la suma de sus costos, menos 1", tol=1e-9)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    ax = _grafico(r, "ax_roi")
    rs = globals().get("roi_por_segmento")
    if ax is not None and isinstance(rs, pd.Series):
        orden = rs.sort_values()
        barras = _barras(ax)
        if len(barras) != len(orden) or not _cerca_lista([b.get_width() for b in barras], orden.tolist()):
            r.mal("`ax_roi` debería tener barras horizontales con `roi_por_segmento`, de menor a mayor (el mayor arriba).")
        elif [_hex(b.get_facecolor()) for b in barras] != [AZUL if x > 0 else GRIS for x in orden]:
            r.mal("En `ax_roi`, pinta en `AZUL` los segmentos con ROI positivo y en `GRIS` los negativos.")
        elif not any(abs(l.get_xdata()[0]) < 1e-12 and len(set(np.atleast_1d(l.get_xdata()))) == 1 for l in ax.get_lines()):
            r.mal("Agrega una línea vertical en 0 (`ax.axvline(0, ...)`): es el punto de equilibrio.")
        else:
            r.ok("`ax_roi` muestra qué segmentos pagan la campaña.")
    elif ax is not None:
        r.mal("Primero resuelve el reto (`roi_por_segmento`).")
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
**`campana`**: una cadena de tiendas envió un correo con un cupón de 10 % a 20 000 clientes... salvo a un **grupo de control** elegido al azar (cerca del 20 %), que no recibió nada. Por cliente: `grupo`, `segmento` (nuevo, regular o frecuente), `canal` preferido, si **abrió** el correo, si hizo **clic**, si **compró** en los 30 días siguientes y el `monto` de esa compra (en soles). En el grupo de control, `abrio` y `clic` son siempre 0.

**`transacciones`**: todas las compras del último año de otros 1500 clientes (`id_cliente`, `fecha`, `monto`), hasta `FECHA_CORTE`.

También tienes `ETAPAS`, `SEGMENTOS` y `SUPUESTOS` (margen de ganancia, costo por cliente contactado y descuento del cupón).

In [ ]:
print(campana.head(), "\n")
print(campana["grupo"].value_counts(), "\n")
print(transacciones.head(), "\n")
print(FECHA_CORTE.date(), SUPUESTOS)

---
## 1. El embudo de conversión

### 📘 Concepto
Un **embudo** sigue a los clientes por las etapas de una acción: recibieron el correo, lo abrieron, hicieron clic y compraron. En cada etapa se pierde gente. Dos tasas lo resumen:
- **conversión por paso**: clientes de una etapa divididos por los de la etapa **anterior**; muestra qué paso falla;
- **conversión total**: clientes de una etapa divididos por los de la **primera**.

La conversión total es el producto de las conversiones por paso. El embudo se dibuja con barras horizontales, la primera etapa arriba, todas en el mismo color.

In [ ]:
etapas_ej = pd.Series({"visitó": 1000, "agregó al carrito": 180, "pagó": 60})
por_paso_ej = etapas_ej / etapas_ej.shift(1)
print(pd.DataFrame({"clientes": etapas_ej, "conv_paso": por_paso_ej.fillna(1.0), "conv_total": etapas_ej / etapas_ej.iloc[0]}))

fig_ej, ax_ej = plt.subplots(figsize=(6, 2.5))
ax_ej.barh(etapas_ej.index[::-1], etapas_ej.values[::-1], color=AZUL)
ax_ej.set_title("Solo 6 de cada 100 visitantes pagan")
plt.show()

### ✍️ Tu turno · Ejercicio 1: el embudo del correo
**Parte A.** Usa solo el grupo **campaña**.
1. `embudo`: un DataFrame con `ETAPAS` como índice y las columnas:
   - `clientes`: cuántos recibieron el correo (todos los del grupo), cuántos lo abrieron, cuántos hicieron clic y cuántos hicieron clic **y** compraron;
   - `conv_paso`: cada etapa dividida por la anterior (1.0 en la primera);
   - `conv_total`: cada etapa dividida por la primera.
2. `fig_embudo, ax_embudo`: barras horizontales con `clientes`, la etapa "recibió" arriba, y la etiqueta `Clientes` en el eje x. Ponle un título que cuente la conclusión.

¿En qué paso se pierden más clientes?

**Parte B.** Predice **sin ejecutar**: si el 30 % abre el correo y el 25 % de quienes lo abren hace clic, ¿qué **porcentaje** de los que lo recibieron hace clic? Guárdalo en `pred_pct_clic` (un número decimal, por ejemplo `12.5`).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Filtra `campana[campana["grupo"] == "campaña"]` y suma las columnas `abrio` y `clic`. Para "compró", suma `(clic == 1) & (compro == 1)`.
</details>

<details><summary>💡 Pista 2</summary>

Arma `clientes` como una Series con `ETAPAS` de índice; `conv_paso` es `clientes / clientes.shift(1)` con `.fillna(1.0)`. Para dibujar con la primera arriba, usa `[::-1]`.
</details>

---
## 2. Incremento y lift: el efecto real de la campaña

### 📘 Concepto
Muchos clientes habrían comprado igual, con o sin correo. Para saber cuánto **causó** la campaña se compara con un **grupo de control**, elegido al azar y que no recibió la acción:
- **incremento**: tasa de compra de la campaña menos la del control (en **puntos porcentuales**, pp);
- **lift**: tasa de la campaña dividida por la del control (1,5 significa 50 % más compras);
- **compras incrementales**: el incremento multiplicado por los clientes contactados.

Contar las compras de quienes hicieron clic (**compras atribuidas**) suele exagerar el efecto: incluye a clientes que igual iban a comprar y abren más los correos. El efecto también puede variar mucho entre segmentos.

In [ ]:
ej = pd.DataFrame({"grupo": ["campaña"] * 6 + ["control"] * 4, "compro": [1, 0, 1, 0, 0, 1, 0, 1, 0, 0]})
tasas_ej = ej.groupby("grupo")["compro"].mean()
print(tasas_ej)
print("lift:", round(tasas_ej["campaña"] / tasas_ej["control"], 2), "| incremento:", round((tasas_ej["campaña"] - tasas_ej["control"]) * 100, 1), "pp")

### ✍️ Tu turno · Ejercicio 2: ¿cuánto vendió de verdad la campaña?
**Parte A.**
1. `tasas`: una Series con la tasa de compra (`compro`) de cada grupo, con índice `campaña` y `control`.
2. `lift` e `incremento_pp`: el lift y el incremento en puntos porcentuales.
3. `compras_incrementales`: el incremento (como proporción, no en pp) multiplicado por la cantidad de clientes del grupo campaña.
4. `compras_atribuidas`: los clientes del grupo campaña que hicieron clic y compraron.
5. `por_segmento`: un DataFrame con `SEGMENTOS` como índice (en ese orden) y las columnas `tasa_control`, `tasa_campana`, `incremento_pp` y `lift`.

¿En qué segmento funciona mejor el correo? ¿En cuál casi no cambia nada?

**Parte B.** Responde en `pred_atribuidas_mayor` con `"sí"` o `"no"`: ¿las compras atribuidas son más que las compras incrementales?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

`campana.groupby("grupo")["compro"].mean()` da las dos tasas. Para los segmentos, agrupa por `["segmento", "grupo"]` y usa `.unstack()`.
</details>

<details><summary>💡 Pista 2</summary>

Después de `unstack()`, las columnas son `campaña` y `control`: renómbralas, calcula las dos columnas nuevas y ordena con `.loc[SEGMENTOS, [...]]`.
</details>

---
## 3. Segmentación RFM

### 📘 Concepto
**RFM** resume el historial de cada cliente con tres números:
- **Recencia**: días desde su última compra hasta la fecha de corte (menos es mejor);
- **Frecuencia**: cuántas compras hizo;
- **Monetario**: cuánto gastó en total.

Cada uno se convierte en un **puntaje de 1 a 5** por quintiles: el 20 % peor recibe 1 y el 20 % mejor, 5. Como hay muchos empates (muchos clientes con 1 compra), primero se ordena con `rank(method="first")` y luego se corta con `pd.qcut`:

```python
pd.qcut(serie.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)                  # mayor es mejor
pd.qcut(serie.rank(method="first", ascending=False), 5, labels=[1, 2, 3, 4, 5]).astype(int)  # menor es mejor
```

Con los puntajes se arman segmentos con reglas simples y accionables (a quién retener, a quién reactivar).

In [ ]:
compras_ej = pd.DataFrame({"id_cliente": [1, 2, 1, 3, 2, 1],
                           "fecha": pd.to_datetime(["2025-01-05", "2025-01-20", "2025-02-01", "2024-11-15", "2025-02-10", "2025-02-20"]),
                           "monto": [30.0, 12.0, 45.0, 80.0, 20.0, 15.0]})
corte_ej = pd.Timestamp("2025-02-28")
resumen_ej = compras_ej.groupby("id_cliente").agg(ultima=("fecha", "max"), compras=("fecha", "count"), gasto=("monto", "sum"))
resumen_ej["dias"] = (corte_ej - resumen_ej["ultima"]).dt.days
print(resumen_ej)
print(pd.qcut(pd.Series([5, 1, 1, 1, 3]).rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int).tolist())

### ✍️ Tu turno · Ejercicio 3: RFM de 1500 clientes
**Parte A.**
1. `calcular_rfm(trans, fecha_corte)`: una función que devuelve un DataFrame con `id_cliente` como índice (ordenado) y las columnas `recencia` (días, entero), `frecuencia` y `monetario`, sin modificar `trans`. Con un DataFrame sin filas, debe devolver uno vacío con esas columnas.
2. `rfm`: `calcular_rfm(transacciones, FECHA_CORTE)` más las columnas `R`, `F` y `M` (puntajes de 1 a 5, con 5 = mejor) y `segmento`, asignado con estas reglas **en este orden** (la primera que se cumple gana):
   - `"campeones"`: R ≥ 4 y F ≥ 4;
   - `"en riesgo"`: R ≤ 2 y F ≥ 3;
   - `"nuevos"`: R ≥ 4 y F ≤ 2;
   - `"dormidos"`: R ≤ 2 y F ≤ 2;
   - `"regulares"`: todos los demás.
3. `resumen_rfm`: un DataFrame con un segmento por fila (índice) y las columnas `clientes` y `monetario_medio`, de más a menos clientes.

¿Qué harías con los clientes "en riesgo"?

**Parte B.** Responde en `pred_recencia_mejor` con `"mayor"` o `"menor"`: ¿qué recencia indica un mejor cliente?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Dentro de la función, copia la idea de `resumen_ej` y termina con `[["recencia", "frecuencia", "monetario"]]`. Para las reglas, `np.select(condiciones, nombres, default="regulares")` respeta el orden.
</details>

<details><summary>💡 Pista 2</summary>

`resumen_rfm = rfm.groupby("segmento").agg(clientes=("R", "size"), monetario_medio=("monetario", "mean")).sort_values("clientes", ascending=False)`.
</details>

---
## 4. ROI de la campaña y supuestos explícitos

### 📘 Concepto
El **retorno de la inversión** (ROI) compara lo que la campaña dejó con lo que costó:

$$ROI = \frac{\text{ganancia incremental} - \text{costo}}{\text{costo}}$$

Para la campaña de hoy, con `n` clientes contactados:
- **ganancia incremental** = `n × (tasa_campaña − tasa_control) × ticket × margen`: solo cuenta lo que se vendió **gracias** a la campaña, y solo el margen, no la venta completa;
- **costo** = `n × costo_contacto + n × tasa_campaña × ticket × descuento`: el cupón se paga en **todas** las compras con cupón, también en las de quienes habrían comprado igual.

Un ROI de 0,2 significa que cada sol invertido volvió con 20 céntimos de ganancia; un ROI negativo, que se perdió dinero. El resultado depende de **supuestos** (margen, costos), así que se escriben y se prueba cuánto cambia el ROI si varían: es un **análisis de sensibilidad**.

In [ ]:
def ganancia_ej(ventas, margen):
    return ventas * margen

tabla_ej = pd.DataFrame({m: [ganancia_ej(v, m) for v in [1000, 2000]] for m in [0.2, 0.3]}, index=[1000, 2000])
tabla_ej.index.name, tabla_ej.columns.name = "ventas", "margen"
print(tabla_ej)

### ✍️ Tu turno · Ejercicio 4: ¿la campaña se pagó sola?
**Parte A.**
1. `roi_campana(n, tasa_campana, tasa_control, ticket, margen, costo_contacto, descuento)`: una función que devuelve el ROI con las fórmulas del concepto. Si el costo es 0, devuelve `None`.
2. `ticket`: el monto promedio de las compras del grupo campaña.
3. `roi_total`: el ROI de la campaña con sus tasas del ejercicio 2, `ticket` y los valores de `SUPUESTOS`.
4. `sensibilidad`: un DataFrame con el ROI para cada margen de `[0.2, 0.3, 0.4, 0.5]` (índice) y cada costo de contacto de `[0.05, 0.15, 0.3]` (columnas), con el descuento de `SUPUESTOS`.
5. `supuestos_campana`: una lista con al menos 3 supuestos de tu cálculo, como frases (por ejemplo, sobre el período medido o sobre los costos que no incluiste).

¿Con qué margen la campaña empezaría a ser rentable?

**Parte B.** Predice **sin ejecutar**: si la campaña no cambia la tasa de compra, ¿cuánto vale el ROI? Guárdalo en `pred_roi_sin_efecto` (un número decimal).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Calcula dentro de la función `ganancia` y `costo` por separado, revisa si el costo es 0 y recién después divide.
</details>

<details><summary>💡 Pista 2</summary>

Para `sensibilidad`, arma un diccionario `{costo: [roi con cada margen]}` y conviértelo con `pd.DataFrame(..., index=margenes)`. Usa `**SUPUESTOS` o `SUPUESTOS["margen"]` para leer los supuestos.
</details>

---
## 🏋️ Reto final: ¿a quién conviene enviarle el correo?
1. `roi_por_segmento`: una Series con el ROI de cada segmento (índice en el orden de `SEGMENTOS`), usando las tasas de control y campaña **de ese segmento**, el ticket de las compras del grupo campaña **de ese segmento** y `SUPUESTOS`.
2. `segmentos_rentables`: la lista de segmentos con ROI positivo, en el orden de `SEGMENTOS`.
3. `roi_focalizado`: el ROI de enviar el correo **solo** a esos segmentos: la suma de sus ganancias incrementales dividida por la suma de sus costos, menos 1.

Escribe en una celda de texto tu recomendación en dos frases, con un número y un supuesto.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Recorre `SEGMENTOS` con un `for`: filtra `campana` por segmento y repite los cálculos del ejercicio 4. Guarda también la ganancia y el costo de cada segmento.
</details>

<details><summary>💡 Pista 2</summary>

`roi_focalizado = sum(ganancias[s] for s in segmentos_rentables) / sum(costos[s] for s in segmentos_rentables) - 1`.
</details>

---
## 🚀 Nivel pro (opcional): el gráfico para la decisión
Crea `fig_roi, ax_roi`: barras horizontales con `roi_por_segmento` ordenado de menor a mayor (el mayor arriba), en `AZUL` los segmentos con ROI positivo y en `GRIS` los negativos, una línea vertical en 0 y un título que diga la recomendación. Si quieres, usa `ax_roi.bar_label(...)` para mostrar cada ROI como porcentaje, `mpl.ticker.PercentFormatter(1.0)` para el eje x y `set_xlim` para que las etiquetas no choquen con los nombres.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P4 · impacto estimado con supuestos explícitos

**Qué hacer**
1. Toma la pregunta 6 del proyecto: *si fueras analista de un banco, ¿dónde atacar primero para reducir reclamos y con qué impacto estimado?* Elige **una** acción concreta sugerida por tus hallazgos de P2 y P3 (por ejemplo, un cambio en el producto o motivo que más reclamos concentra).
2. En un notebook nuevo (o una sección al final de `04_modelo.ipynb`), arma una tabla de **supuestos**: cuántos reclamos afecta la acción, qué porcentaje podría reducir, cuánto cuesta atender un reclamo y cuánto cuesta la acción. Cita de dónde sale cada número o marca que es un supuesto tuyo.
3. Calcula el impacto con una función, como `roi_campana` hoy, y una tabla de **sensibilidad** con un escenario pesimista, uno base y uno optimista.
4. Escribe en el README una sección **Recomendación** en borrador: la acción, el impacto en el escenario base, el rango entre pesimista y optimista y qué dato haría falta para confirmarlo (idealmente, un piloto con grupo de control).

**Por qué lo haría un analista**
Quien decide no compra un número, compra un razonamiento. Mostrar los supuestos y cuánto cambia el resultado si fallan hace que la recomendación sea discutible y, por eso, creíble.

**Cómo debe verse el resultado**
Una tabla de supuestos con su fuente, una función que calcula el impacto, una tabla de escenarios y un párrafo de recomendación que alguien sin conocimientos técnicos entiende en un minuto. Lo usarás en la presentación final (S27).

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Construir un embudo y explicar la diferencia entre conversión por paso y total.
- [ ] Explicar por qué hace falta un grupo de control y calcular incremento y lift.
- [ ] Explicar por qué las compras atribuidas no son el efecto de la campaña.
- [ ] Calcular recencia, frecuencia y monto, convertirlos en puntajes y segmentar.
- [ ] Calcular un ROI, escribir sus supuestos y hacer un análisis de sensibilidad.
- [ ] Recomendar a quién dirigir una campaña con un número y un supuesto.

**Próxima sesión (S24):** aprendizaje no supervisado: K-means, cómo elegir el número de grupos, PCA e interpretación de segmentos.